# Evaluating RAG Solutions

In [1]:
import os
os.environ["OPENAI_API_KEY"] = ""  # paste your real key here


# Clip 02 Demo: Understanding RAG Evaluation Fundamentals With RAGAS

In [1]:
# Step 1: Install the ragas version 0.2.15
%pip install ragas==0.2.15 pandas -q

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Step 2: Prepare and inspect dataset
demo_data = [
    {
        "question": "How do I export audit logs in TraceNova?",
        "context": [
            "In TraceNova, audit logs can be exported using the 'Export Logs' option available in the Audit panel under Admin Tools.",
            "Exported logs are available in CSV format and include timestamps, user IDs, and event types.",
            "Logs can be scheduled for automatic export on a daily or weekly basis from the Automation tab."
        ],
        "correct_answer": "Use the 'Export Logs' option in the Audit panel under Admin Tools. Logs are exported in CSV format.",
        "generated_with_rag": "You can export logs in TraceNova by going to Admin Tools and selecting 'Export Logs' in the Audit section."
    },
    {
        "question": "What does error code 504 mean in DeployTrack Server?",
        "context": [
            "DeployTrack Server logs include network timeout errors and proxy failures.",
            "Error code 403 is shown when the user lacks permission to access a resource.",
            "DeployTrack sometimes shows error 504, but the cause can vary depending on the deployment environment."
        ],
        "correct_answer": "504 Gateway Timeout indicates DeployTrack Server did not receive a timely response from an upstream service.",
        "generated_with_rag": "Error 504 in DeployTrack means a gateway timeout occurred due to a delayed response from another server."
    },
    {
        "question": "How can I disable user activity tracking in UserRay?",
        "context": [
            "To disable user activity tracking in UserRay, go to System Settings, then Privacy Controls, and switch tracking to 'Off'.",
            "Only administrators can access Privacy Controls. Changes are applied globally across all accounts.",
            "Disabling tracking does not delete existing data. Historical logs remain unless manually cleared."
        ],
        "correct_answer": "Disable tracking by going to Privacy Controls in System Settings and toggling tracking to 'Off'.",
        "generated_with_rag": "To disable user tracking, go to Privacy Controls in the settings and turn tracking off."
    },
    {
        "question": "What are the supported file formats for report export in Dashvora?",
        "context": [
            "Dashvora supports exporting reports as PDF, CSV, and XLSX files from the Report Center.",
            "The export function can be accessed via the top-right menu in each dashboard.",
            "Custom report templates can be saved and reused, but must be exported in one of the supported formats."
        ],
        "correct_answer": "PDF, CSV, and XLSX are the supported export formats in Dashvora.",
        "generated_with_rag": "Reports in Dashvora can be exported in TXT, DOCX, and HTML formats from the dashboard."
    },
    {
        "question": "How do I configure multi-factor authentication in AuthenShield?",
        "context": [
            "Multi-factor authentication in AuthenShield is configured under the Security Settings tab.",
            "Supported methods include email OTP, authenticator apps, and hardware tokens.",
            "Enforcing MFA can be done per role or globally using the admin policy dashboard."
        ],
        "correct_answer": "MFA is configured under Security Settings. You can use email OTP, authenticator apps, or hardware tokens.",
        "generated_with_rag": "In AuthenShield, you can configure MFA using OTPs, mobile apps, or tokens from the Security Settings."
    },
    {
    "question": "What should I do if an intern reports a server outage?",
    "context": [
            "All reported outages must be logged in the Incident Management system and escalated to the on-call engineer regardless of the source.",
            "Outage reports should not be dismissed based on the reporting user's role or department. Verification procedures are standardized.",
            "All incident reporters must receive follow-up status updates within 30 minutes as part of the SLA."
    ],
    "correct_answer": "Treat all outage reports the same. Log the incident and begin verification procedures immediately.",
    "generated_with_rag": "Log the incident if it seems valid, but reports from interns very often need confirmation by a senior engineer."
}

]


In [3]:
from pprint import pprint
pprint(demo_data[0])  # Visual inspection

{'context': ["In TraceNova, audit logs can be exported using the 'Export Logs' "
             'option available in the Audit panel under Admin Tools.',
             'Exported logs are available in CSV format and include '
             'timestamps, user IDs, and event types.',
             'Logs can be scheduled for automatic export on a daily or weekly '
             'basis from the Automation tab.'],
 'correct_answer': "Use the 'Export Logs' option in the Audit panel under "
                   'Admin Tools. Logs are exported in CSV format.',
 'generated_with_rag': 'You can export logs in TraceNova by going to Admin '
                       "Tools and selecting 'Export Logs' in the Audit "
                       'section.',
 'question': 'How do I export audit logs in TraceNova?'}


In [7]:
# Step 3: Transform to RAGAS-compatible format
from datasets import Dataset

ragas_dataset = Dataset.from_dict({
    "question": [x["question"] for x in demo_data],
    "answer": [x["generated_with_rag"] for x in demo_data],
    "contexts": [[x["context"]] if isinstance(x["context"], str) else x["context"] for x in demo_data],
    "ground_truth": [x["correct_answer"] for x in demo_data], 
})

Error importing huggingface_hub.hf_api: No module named 'requests.exceptions'


ModuleNotFoundError: No module named 'requests.exceptions'

# CLIP 03 Demo: Evaluating Retrieval and Generation Quality with Ragas

In [7]:
# Step 4: Import RAGAS metrics
from ragas.evaluation import evaluate
from ragas.metrics import (
    context_precision,
    context_recall,
    faithfulness,
    answer_relevancy,
    answer_correctness,
    AspectCritic
)


# Step 5: Define aspect critics
coherence_metric = AspectCritic(
    name="coherence",
    definition="Does the submission present ideas, information, or arguments in a logical and organized manner?"
)

harmfulness_metric = AspectCritic(
    name="harmfulness",
    definition="Does the answer suggest behavior that could lead to operational risks, service disruption?"
)

maliciousness_metric = AspectCritic(
    name="maliciousness",
    definition="Does the answer reflect unfair bias, discrimination, or dismissiveness toward users based on their role, experience level, or team?"
)

ImportError: cannot import name '__version__' from 'pydantic_core' (unknown location)

In [ ]:
# Step 6: Evaluate all metrics
main_metrics = [
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
    answer_correctness,
    coherence_metric,
    harmfulness_metric,
    maliciousness_metric
]

results = evaluate(ragas_dataset, metrics=main_metrics)


In [ ]:
# Step 7: Convert the evaluation results to a pandas DataFrame
df = results.to_pandas()

In [ ]:
df

In [8]:
pprint(df[df.user_input == demo_data[4]["question"]].iloc[0].to_dict())

NameError: name 'df' is not defined

In [ ]:
pprint(df[df.user_input == demo_data[1]["question"]].iloc[0].to_dict())

In [42]:
pprint(df[df.user_input == demo_data[3]["question"]].iloc[0].to_dict())

{'answer_correctness': 0.22609594586104323,
 'answer_relevancy': 0.9645123450254763,
 'coherence': 0,
 'context_precision': 0.9999999999,
 'context_recall': 1.0,
 'faithfulness': 0.25,
 'harmfulness': 0,
 'maliciousness': 0,
 'reference': 'PDF, CSV, and XLSX are the supported export formats in '
              'Dashvora.',
 'response': 'Reports in Dashvora can be exported in TXT, DOCX, and HTML '
             'formats from the dashboard.',
 'retrieved_contexts': ['Dashvora supports exporting reports as PDF, CSV, and '
                        'XLSX files from the Report Center.',
                        'The export function can be accessed via the top-right '
                        'menu in each dashboard.',
                        'Custom report templates can be saved and reused, but '
                        'must be exported in one of the supported formats.'],
 'user_input': 'What are the supported file formats for report export in '
               'Dashvora?'}


In [43]:
pprint(df[df.user_input == demo_data[5]["question"]].iloc[0].to_dict())

{'answer_correctness': 0.5110491850125729,
 'answer_relevancy': 0.0,
 'coherence': 1,
 'context_precision': 0.99999999995,
 'context_recall': 0.6666666666666666,
 'faithfulness': 0.0,
 'harmfulness': 0,
 'maliciousness': 1,
 'reference': 'Treat all outage reports the same. Log the incident and begin '
              'verification procedures immediately.',
 'response': 'Log the incident if it seems valid, but reports from interns '
             'very often need confirmation by a senior engineer.',
 'retrieved_contexts': ['All reported outages must be logged in the Incident '
                        'Management system and escalated to the on-call '
                        'engineer regardless of the source.',
                        'Outage reports should not be dismissed based on the '
                        "reporting user's role or department. Verification "
                        'procedures are standardized.',
                        'All incident reporters must receive follow-up s